# What the hell is FP??

## Converting to Floating Point Representation

### Step 1: Original Number
$13.25$

### Step 2: Convert to Binary
$1101.01$

### Step 3: Normalize (Scientific Notation)
$1.10101 \times 2^3$

### Step 4: Extract Components
* **Sign** = 0
* **Exponent** = 3
* **Mantissa** = 10101

## What is a Floating Point Number?

A floating-point number is just a way of storing real numbers (numbers with decimals).

Instead of storing the entire decimal number directly, computers store it in **scientific notation**.

### Base 10 Example

The number $12345$ can be written as:

$$1.2345 \times 10^4$$

### Base 2 (Binary) Example

Computers use the same concept, but with base 2 instead of base 10.

**Original:** $20$

**Binary:** $10100$

**Scientific binary:** $1.01 \times 2^4$

```python
Value = (-1)^Sign × Mantissa × 2^Exponent
```

## The Three Components of a Floating Point Number

### Example: Storing 13.25

**Binary representation:** $1101.01$

**Normalize it:** $1.10101 \times 2^3$

**Components extracted:**
* Sign = 0
* Exponent = 3
* Mantissa = 10101

These three fields are what every FP format stores.

---

### Component 1: Sign Bit

Very simple.

* **0** = Positive
* **1** = Negative

**Examples:**
* $+5$ → Sign = 0
* $-5$ → Sign = 1

Only one bit is needed.

---

### Component 2: Exponent

Exponent decides the **range** of representable numbers.

**Example:** With different exponents, we can represent:
* $1 \times 2^0 = 1$
* $1 \times 2^1 = 2$
* $1 \times 2^2 = 4$
* $1 \times 2^3 = 8$
* $1 \times 2^4 = 16$

The larger the exponent, the larger numbers you can represent.

---

### Component 3: Mantissa

Mantissa (also called **significand** or **fraction**) decides the **precision**.

**Example:** Different mantissa values:
* $1.000 = 1$
* $1.001 = 1.125$
* $1.010 = 1.25$

More mantissa bits = finer precision.

## FP32 Bit Layout

```
+---+--------+-----------------------+
| S | Exp(8) | Mantissa (23)         |
+---+--------+-----------------------+
```

* **1 Sign bit**
* **8 Exponent bits**
* **23 Mantissa bits**
* **Total: 32 bits**

## Example: Storing 13.25 in FP32

**Original number:** $13.25$

**Binary:** $1101.01$

**Normalize:** $1.10101 \times 2^3$

**FP32 stores approximately:**
* **Sign:** 0
* **Exponent:** 3 (biased internally)
* **Mantissa:** 10101000000000000000000 (padded to 23 bits)

## FP16 (Half Precision)

### Reduce Everything

When we move to FP16, we shrink the storage by half:

**FP16 Bit Layout:**
* **1 Sign bit**
* **5 Exponent bits**
* **10 Mantissa bits**
* **Total: 16 bits**

## Example: FP16 Precision Loss

### Storing 13.25

The number $13.25$ is still representable in FP16.

However, now we only have **10 mantissa bits** instead of 23.

**Precision comparison:**
* **Exact value:** $3.1415926535$
* **FP16 stores:** $3.140625$

Notice the precision loss with fewer mantissa bits.

## BF16 (Brain Floating Point)

### Why Google Designed BF16

Google noticed something important:

**Neural networks often need large range, but not necessarily high precision.**

### The Strategy

BF16 keeps the same exponent size as FP32 while shrinking the mantissa.

**BF16 Bit Layout:**
* **1 Sign bit**
* **8 Exponent bits** ← Same as FP32!
* **7 Mantissa bits**
* **Total: 16 bits**

```
+---+--------+---------+
| S | Exp(8) | Man(7)  |
+---+--------+---------+
```

### Comparison: FP16 vs BF16

| Format | Exponent Bits | Mantissa Bits |
|--------|---------------|---------------|
| FP16   | 5             | 10            |
| BF16   | 8             | 7             |

### Why is This Useful?

Suppose your values are:

* $0.0000001$ (very small)
* $10$ (medium)
* $100000$ (very large)

**FP16 Problem:** May overflow or underflow because its exponent range is limited.

**BF16 Solution:** Can represent all of these because it inherits FP32's exponent range.

**Trade-off:** The decimal precision is lower (only 7 mantissa bits).

### Why BF16 Became the Standard

This is why BF16 has become the default training format on modern accelerators like:
* NVIDIA Hopper
* NVIDIA Blackwell
* Google's TPUs

## FP8 and FP4 (Aggressive Quantization)

### FP8 (8-bit Floating Point)

#### Entering Aggressive Optimization

Now we enter the world of serious model compression.

#### Layout (One Common Variant: E4M3)

```
+---+------+------+
| S | Exp  | Man  |
+---+------+------+
```

* **1 Sign bit**
* **4 Exponent bits**
* **3 Mantissa bits**
* **Total: 8 bits**

#### Memory Savings

**Example:** 100 Million weights

$$100\text{M} \times 1\text{ byte} = 100\text{ MB}$$

That's a **4× reduction** compared to FP32 (which would be 400 MB).

#### What Do You Lose?

**Very fine precision.** Example:

* Exact: $1.23456$
* FP8 stores: $1.25$

Another example:
* Exact: $0.117$
* FP8 stores: $0.125$

These rounding errors are much larger than in FP16, but many neural network layers tolerate them surprisingly well, especially during inference.

---

### FP4 (4-bit Floating Point)

#### The Extreme Challenge

This is where things become very constrained.

You only have **4 bits total**.

A conceptual layout is:

```
+---+----+---+
| S |Exp | M |
+---+----+---+
```

* **1 Sign bit**
* **2 Exponent bits**
* **1 Mantissa bit**

#### Limited Representable Values

There are only:

$$2^4 = 16 \text{ possible bit patterns}$$

That means the format can represent only a tiny set of values.

**Imagine (illustratively):**
* $0$
* $\pm 0.5$
* $\pm 1$
* $\pm 2$
* $\pm 4$
* $\pm 8$
* ...

#### Extreme Precision Loss

**Try storing $1.234$:**
* Becomes either $1$ or $1.5$ (depending on the encoding)

**Try storing $0.093$:**
* May become $0$ or $0.125$

The quantization error is **much larger** than FP8.

#### When FP4 is Actually Used

This is why **plain FP4 is rarely used by itself**. Instead, it is paired with **block scaling**, which:
* Rescales groups of values
* Allows FP4 to represent them much more accurately
* Dramatically reduces quantization error

## Example: Precision Loss Across Formats

### Original Value: π

$$\pi = 3.14159265...$$

| Format |               Stored Value (illustrative) |
| ------ | ----------------------------------------: |
| FP32   |                                 3.1415927 |
| FP16   |                                  3.140625 |
| BF16   | 3.140625 (similar range, lower precision) |
| FP8    |                                     3.125 |
| FP4    |                                3.0 or 4.0 |


## The Fundamental Trade-Off

As engineers, the key trade-off is always:

### More Bits

* ✓ More memory
* ✓ More bandwidth consumption
* ✓ Higher precision (less quantization error)

### Fewer Bits

* ✓ Less memory
* ✓ Faster computation
* ✗ More quantization error

**The goal:** Find the sweet spot where model accuracy remains acceptable while gaining speed and memory savings.